# Community Notes 陣営反応分析（unzip_data 版）

Drive の `基礎プロジェクト/unzip_data/` から解凍済み TSV を直接コピーして使う実験版です。
安定版は `colab_full_run.ipynb` を使用してください。

上から順にセルを実行してください。
「★ 設定」セルのパラメータを変えることで、実行時間やトピックを調整できます。

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ 設定: ここを変えて実行時間・対象を調整 ★
# ====================================================

# --- Drive のパス ---
DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'

# --- 結果の保存先 ---
# Shared Drives は読み取り専用の場合があるため、MyDrive に保存
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results'

# --- ratings ファイルの読み込み数 ---
# 全8ファイル → 最も正確だが時間がかかる（推定1〜2時間）
# 2ファイル  → 1/4のデータで高速（推定15〜30分）、大体の傾向がわかる
# 1ファイル  → 最速テスト（推定5〜10分）
RATINGS_FILES = 2       # ← 1〜8 の数字を入れる。全部なら 8

# --- ratings の行数制限 ---
# None  → ファイル全量読み込み（推奨）
# 500000 → 各ファイルの先頭50万行のみ（さらに高速テスト）
NROWS = None            # ← None または数字

# --- 分析対象トピック ---
# キーワードを自由に追加・削除・変更できます
# トピックを減らすと実行が速くなります
TOPICS = {
    'vaccine_covid': [
        'vaccine', 'covid', 'pandemic', 'mask', 'booster',
        'pfizer', 'moderna', 'antivax', 'lockdown', 'fauci',
    ],
    'israel_palestine': [
        'israel', 'palestine', 'gaza', 'hamas', 'idf',
        'netanyahu', 'ceasefire', 'zionist', 'hezbollah',
    ],
    'trump': [
        'trump', 'maga', 'indictment', 'mar-a-lago',
        'january 6', 'j6', 'impeach',
    ],
    'immigration': [
        'immigration', 'border', 'migrant', 'asylum',
        'deportation', 'illegal alien', 'refugee', 'caravan',
    ],
    'gun_control': [
        'gun control', 'second amendment', '2nd amendment',
        'shooting', 'nra', 'firearm', 'gun violence',
    ],
    'ALL_POLITICAL': [
        'trump', 'biden', 'democrat', 'republican', 'gop',
        'congress', 'senate', 'election', 'vote', 'ballot',
        'liberal', 'conservative', 'immigration', 'border',
        'abortion', 'gun control', 'vaccine', 'covid',
        'climate change', 'supreme court',
        'israel', 'palestine', 'gaza', 'ukraine', 'russia',
        'woke', 'dei', 'transgender', 'lgbtq', 'censorship',
        'government', 'policy', 'partisan',
    ],
}

# =============================================
# ★ 分析パラメータ（上級者向け）
# =============================================
# 通常はデフォルト値で問題ありません。
# 分析の感度や閾値を細かく調整したい場合に変更してください。

# --- 前処理: Polarity 計算 ---
# 各評価者（rater）の政治的立ち位置（polarity）を推定するために
# 使用する評価数。最初の N 件の評価のみを使い、それ以降の評価には
# polarity を「固定」して適用することで、循環論法を回避する。
# 大きくすると推定精度が上がるが、固定される前の評価が増える。
POLARITY_FIRST_N = 50   # ← 推奨: 30〜100

# --- 前処理: ノートのフィルタリング ---
# 評価数がこの値未満のノートは分析対象から除外される。
# 小さくするとノート数が増えるがノイズも増える。
# run_pipeline（全体分析）で使用される。
MIN_RATING_COUNT = 20    # ← 推奨: 10〜50

# --- 前処理: トピック別分析のフィルタ ---
# トピック別分析（run_by_topic）でノートに必要な最低評価数。
# トピックごとのデータ量は全体より少ないため、
# MIN_RATING_COUNT より低めに設定するのが一般的。
MIN_RATINGS_TOPIC = 10   # ← 推奨: 5〜30

# --- バースト検出: 速度倍率 ---
# ノートの平均評価速度に対して、何倍以上の速度が観測されたら
# 「バースト」（評価の急激な集中）とみなすか。
# 小さくするとバーストが多く検出され、大きくすると厳選される。
BURST_SPEED_MULTIPLIER = 3.0  # ← 推奨: 2.0〜5.0

# --- バースト検出: 最小評価数 ---
# バーストと判定するために、集中区間内に最低限必要な評価の数。
# スライディングウィンドウのサイズにもなる。
# 小さくすると小規模なバーストも検出される。
BURST_MIN_COUNT = 5      # ← 推奨: 3〜10

# --- バースト分類: TypeA/TypeB 閾値 ---
# バースト内の評価者の polarity 分散に基づき、
# TypeA（陣営反応: 同じ立場の人が集中）と
# TypeB（自然拡散: 多様な立場の人が集まる）に分類する閾値。
# None → 全バーストの中央値で自動決定（推奨）
# 数値を指定すると、その値以下を TypeA、超を TypeB に分類。
BURST_THRESHOLD = None   # ← None（自動）または 0.0〜1.0 程度の数値

# --- 特徴量: トレンド計算の最小評価数 ---
# ノートの評価推移（前半 vs 後半の平均スコアの差）を計算するために
# 必要な最低評価数。これ未満のノートはトレンド=0 として扱われる。
TREND_MIN_EVALS = 4      # ← 推奨: 3〜10

# --- ターゲット抽出: 品質スコア上位 % ---
# 「正しいのに消えた」ノートを特定するために、
# 品質スコア（URLの数・文字数から算出）の上位何%を対象にするか。
# 小さくするとより高品質なノートのみに絞られる。
TARGET_TOP_PERCENT = 25  # ← 推奨: 10〜50

# ====================================================
print(f'ratings files: {RATINGS_FILES}/8')
print(f'nrows: {NROWS if NROWS else "全量"}')
print(f'topics: {list(TOPICS.keys())}')
print(f'save dir: {SAVE_DIR}')
print()
print('--- 分析パラメータ ---')
print(f'polarity_first_n:       {POLARITY_FIRST_N}')
print(f'min_rating_count:       {MIN_RATING_COUNT}')
print(f'min_ratings_topic:      {MIN_RATINGS_TOPIC}')
print(f'burst_speed_multiplier: {BURST_SPEED_MULTIPLIER}')
print(f'burst_min_count:        {BURST_MIN_COUNT}')
print(f'burst_threshold:        {BURST_THRESHOLD if BURST_THRESHOLD is not None else "自動（中央値）"}')
print(f'trend_min_evals:        {TREND_MIN_EVALS}')
print(f'target_top_percent:     {TARGET_TOP_PERCENT}%')

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, json

if os.path.exists(DRIVE_DATA_UNZIPPED):
    print('OK: unzip_data フォルダが見つかりました')
    for d in sorted(os.listdir(DRIVE_DATA_UNZIPPED)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA_UNZIPPED} が見つかりません')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('共有ドライブ一覧:')
        for d in os.listdir(sd): print(f'  {d}')
    print('マイドライブ:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]: print(f'  {d}')

In [ ]:
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pip install -q statsmodels

In [ ]:
# data/raw/ にデータを準備（unzip_data から cp、あればスキップ）
raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

def _copy_if_missing(src_path, dst_path):
    if os.path.exists(dst_path):
        print(f'  {os.path.basename(dst_path)} already exists, skip')
        return
    print(f'  Copying {os.path.basename(src_path)} ...')
    subprocess.run(['cp', src_path, dst_path], check=True)

# --- ratings: 必要な分だけコピー、余分は削除 ---
ratings_folder = os.path.join(DRIVE_DATA_UNZIPPED, 'RatingData')
if os.path.exists(ratings_folder):
    all_tsvs = sorted([f for f in os.listdir(ratings_folder) if f.startswith('ratings-') and f.endswith('.tsv')])
    needed = set(all_tsvs[:RATINGS_FILES])
    # 余分な ratings tsv を削除
    for f in sorted(os.listdir(raw_dir)):
        if f.startswith('ratings-') and f.endswith('.tsv') and f not in needed:
            print(f'  Removing {f} (over limit {RATINGS_FILES})')
            os.remove(os.path.join(raw_dir, f))
    for tsv in all_tsvs[:RATINGS_FILES]:
        _copy_if_missing(os.path.join(ratings_folder, tsv), os.path.join(raw_dir, tsv))
else:
    print(f'WARNING: {ratings_folder} not found')

# --- notes, history: あればスキップ ---
other_targets = [
    ('Notes data', 'notes-'),
    ('Note status history data', 'noteStatusHistory-'),
]
for subfolder, prefix in other_targets:
    folder = os.path.join(DRIVE_DATA_UNZIPPED, subfolder)
    if not os.path.exists(folder):
        print(f'WARNING: {folder} not found, skip')
        continue
    for f in sorted(os.listdir(folder)):
        if f.startswith(prefix) and f.endswith('.tsv'):
            _copy_if_missing(os.path.join(folder, f), os.path.join(raw_dir, f))

print()
print('=== data/raw/ ===')
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.tsv'):
        size = os.path.getsize(os.path.join(raw_dir, f)) / (1024**3)
        print(f'  {f}  ({size:.1f} GB)')

---
## 1. 全トピックでパイプライン実行

In [ ]:
nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_pipeline.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --polarity-first-n {POLARITY_FIRST_N} \
    --min-rating-count {MIN_RATING_COUNT} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS} \
    --target-top-percent {TARGET_TOP_PERCENT}

---
## 2. トピック別比較

In [ ]:
with open('/tmp/topics.json', 'w') as f:
    json.dump(TOPICS, f)

nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_by_topic.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --min-ratings {MIN_RATINGS_TOPIC} \
    --topics-json /tmp/topics.json \
    --polarity-first-n {POLARITY_FIRST_N} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS}

---
## 3. 結果の確認

In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/topic_comparison.csv')
display(df)

In [ ]:
targets = pd.read_csv('data/processed/target_notes.csv')
print(f'ターゲットノート数: {len(targets)}')
display(targets.head(20))

In [ ]:
bursts = pd.read_csv('data/processed/bursts.csv')
print(f'バースト数: {len(bursts)}')
print(bursts['burst_type'].value_counts())

---
## 4. 結果を Drive に保存

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
for f in os.listdir('data/processed'):
    src = os.path.join('data/processed', f)
    if f.endswith('.csv') or f.endswith('.png'):
        subprocess.run(['cp', src, SAVE_DIR])
print(f'Done! Results saved to: {SAVE_DIR}')